# ADAS Vision — Phase 6c: Train on the FULL IDD Segmentation Dataset

Trains the drivable-area model on the complete Indian Driving Dataset (~20k full-resolution images) instead of IDD-Lite. More data + higher resolution = the accurate model.

**Designed to survive Colab free-tier disconnects:** every epoch checkpoints to your Drive and the notebook auto-resumes — if the session dies, just Runtime → Run all again.

## One-time setup
1. Register (free) at https://idd.insaan.iiit.ac.in/
2. Download **IDD Segmentation (IDD 20k Part I)** (~18 GB). *(Part II ~3 GB is optional extra data — the notebook uses it automatically if present.)*
3. Upload the archive(s) to Drive: `MyDrive/adas/` (keep original names, e.g. `idd-segmentation.tar.gz` / any `*.tar.gz` containing `leftImg8bit` + `gtFine`)
4. Keep `MyDrive/adas/dashcam.mp4` there too
5. Runtime → Change runtime type → **T4 GPU**

**Timeline:** extraction ~10 min → mask generation ~15 min (first session only, cached to Drive) → training ~8 min/epoch, 24 epochs ≈ 3.5 hrs. One to two sessions.

> Advisory/research only — never connect any of this to a vehicle's controls.

In [ ]:
# 1) GPU + Drive
!nvidia-smi -L
import torch, os, glob, tarfile, json, time
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

from google.colab import drive
drive.mount("/content/drive")
ADAS = "/content/drive/MyDrive/adas"
CKPT = os.path.join(ADAS, "phase6c_ckpt.pth")          # resumable checkpoint on Drive
BEST = os.path.join(ADAS, "drivable_idd_full_best.pth")
MASK_CACHE = os.path.join(ADAS, "idd_masks_cache.tar")  # rasterized labels cache

In [ ]:
# 2) Extract the dataset to FAST local disk (never train off the Drive mount)
DATA = "/content/idd"
if not glob.glob(os.path.join(DATA, "**", "leftImg8bit"), recursive=True):
    os.makedirs(DATA, exist_ok=True)
    archives = [p for p in glob.glob(os.path.join(ADAS, "*.tar.gz")) + glob.glob(os.path.join(ADAS, "*.tgz"))
                if "lite" not in os.path.basename(p).lower()]
    assert archives, "No full-IDD archive found in MyDrive/adas/ — upload the 18GB tar.gz there"
    for arc in archives:
        print("extracting", os.path.basename(arc), "... (several minutes)")
        t0 = time.time()
        with tarfile.open(arc) as t:
            t.extractall(DATA)
        print(f"  done in {time.time()-t0:.0f}s")

img_root_hits = glob.glob(os.path.join(DATA, "**", "leftImg8bit"), recursive=True)
assert img_root_hits, "extracted, but no leftImg8bit folder found"
ROOT = os.path.dirname(img_root_hits[0])
print("dataset root:", ROOT)
n_train_imgs = len(glob.glob(os.path.join(ROOT, "leftImg8bit", "train", "*", "*.*")))
n_val_imgs   = len(glob.glob(os.path.join(ROOT, "leftImg8bit", "val",   "*", "*.*")))
print(f"train images: {n_train_imgs} | val images: {n_val_imgs}")

In [ ]:
# 3) Rasterize polygon labels -> binary drivable masks (cached to Drive)
# Full IDD ships gtFine as *_polygons.json. Drivable = road / parking /
# drivable fallback (IDD level-1 'drivable' group). Masks are written at
# HALF resolution (960x540) — plenty for segmentation training, 4x smaller.
import numpy as np
from PIL import Image, ImageDraw
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor

MASKS = "/content/masks"
DRIVABLE_LABELS = {"road", "parking", "drivable fallback"}
MW, MH = 960, 540

def rasterize_one(args):
    jp, outp = args
    try:
        with open(jp) as f:
            d = json.load(f)
        w, h = d.get("imgWidth", 1920), d.get("imgHeight", 1080)
        m = Image.new("L", (MW, MH), 0)
        draw = ImageDraw.Draw(m)
        sx, sy = MW / w, MH / h
        for obj in d.get("objects", []):
            if obj.get("deleted"):
                continue
            if str(obj.get("label", "")).lower() in DRIVABLE_LABELS:
                poly = [(x * sx, y * sy) for x, y in obj.get("polygon", [])]
                if len(poly) >= 3:
                    draw.polygon(poly, fill=255)
        os.makedirs(os.path.dirname(outp), exist_ok=True)
        m.save(outp)
        return True
    except Exception as e:
        return f"{jp}: {e}"

if os.path.exists(MASK_CACHE) and not os.path.isdir(MASKS):
    print("restoring mask cache from Drive ...")
    with tarfile.open(MASK_CACHE) as t:
        t.extractall("/content")

if not os.path.isdir(MASKS):
    jobs = []
    for split in ("train", "val"):
        for jp in glob.glob(os.path.join(ROOT, "gtFine", split, "*", "*_polygons.json")):
            rel = os.path.relpath(jp, os.path.join(ROOT, "gtFine"))
            outp = os.path.join(MASKS, rel.replace("_gtFine_polygons.json", "_drivable.png")
                                          .replace("_polygons.json", "_drivable.png"))
            jobs.append((jp, outp))
    print(f"rasterizing {len(jobs)} label files ...")
    errs = []
    with ProcessPoolExecutor(max_workers=4) as ex:
        for r in tqdm(ex.map(rasterize_one, jobs, chunksize=64), total=len(jobs)):
            if r is not True:
                errs.append(r)
    print(f"done, {len(errs)} errors")
    if errs[:3]:
        print(errs[:3])
    print("caching masks to Drive for future sessions ...")
    with tarfile.open(MASK_CACHE, "w") as t:
        t.add(MASKS, arcname="masks")
print("masks ready")

In [ ]:
# 4) Pair images with masks
import cv2, random
from torch.utils.data import Dataset, DataLoader

def pairs(split):
    out = []
    for ip in sorted(glob.glob(os.path.join(ROOT, "leftImg8bit", split, "*", "*.*"))):
        base = os.path.splitext(os.path.basename(ip))[0].replace("_leftImg8bit", "").replace("_image", "")
        seq = os.path.basename(os.path.dirname(ip))
        cands = glob.glob(os.path.join(MASKS, split, seq, base + "*_drivable.png"))
        if cands:
            out.append((ip, cands[0]))
    return out

train_pairs, val_pairs = pairs("train"), pairs("val")
print(f"train: {len(train_pairs)} | val: {len(val_pairs)}")
assert len(train_pairs) > 1000, "pairing looks wrong — inspect the mask folder layout above"

IN_W, IN_H = 768, 432
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)

class IDDFull(Dataset):
    def __init__(self, pair_list, train=True):
        self.pairs, self.train = pair_list, train
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        ip, mp = self.pairs[i]
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        lbl = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IN_W, IN_H))
        lbl = cv2.resize(lbl, (IN_W, IN_H), interpolation=cv2.INTER_NEAREST)
        if self.train:
            if random.random() < 0.5:
                img, lbl = img[:, ::-1], lbl[:, ::-1]
            if random.random() < 0.4:   # shadow/brightness robustness (hill roads!)
                img = np.clip(img.astype(np.float32) * random.uniform(0.5, 1.5), 0, 255)
        x = (img.astype(np.float32) / 255.0 - MEAN) / STD
        x = torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))
        y = torch.from_numpy(np.ascontiguousarray((lbl > 127).astype(np.int64)))
        return x, y

train_dl = DataLoader(IDDFull(train_pairs, True),  batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(IDDFull(val_pairs,  False), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
print("dataloaders ready")

In [ ]:
# 5) Model + RESUMABLE training (checkpoint to Drive every epoch)
import torchvision
from torch import nn

EPOCHS = 24
model = torchvision.models.segmentation.lraspp_mobilenet_v3_large(weights="DEFAULT")
model.classifier.low_classifier  = nn.Conv2d(40, 2, 1)
model.classifier.high_classifier = nn.Conv2d(128, 2, 1)
model = model.to(DEVICE)

opt = torch.optim.AdamW([
    {"params": model.backbone.parameters(),   "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-3},
], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()
lossf = nn.CrossEntropyLoss()

start_ep, best_iou = 0, 0.0
if os.path.exists(CKPT):                      # ── auto-resume after a disconnect
    ck = torch.load(CKPT, map_location=DEVICE)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    start_ep, best_iou = ck["epoch"] + 1, ck["best_iou"]
    print(f"resumed from epoch {start_ep} (best IoU so far {best_iou:.3f})")

def val_iou():
    model.eval(); inter = union = 0
    with torch.no_grad():
        for x, y in val_dl:
            p = model(x.to(DEVICE, non_blocking=True))["out"].argmax(1).cpu()
            inter += ((p == 1) & (y == 1)).sum().item()
            union += ((p == 1) | (y == 1)).sum().item()
    return inter / max(union, 1)

for ep in range(start_ep, EPOCHS):
    model.train(); running = 0.0
    for x, y in tqdm(train_dl, desc=f"epoch {ep+1}/{EPOCHS}"):
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss = lossf(model(x.to(DEVICE, non_blocking=True))["out"], y.to(DEVICE, non_blocking=True))
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        running += loss.item() * x.size(0)
    sched.step()
    iou = val_iou()
    print(f"epoch {ep+1}: loss {running/len(train_pairs):.4f} | val drivable-IoU {iou:.3f}")
    if iou > best_iou:
        best_iou = iou
        torch.save(model.state_dict(), BEST)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                "epoch": ep, "best_iou": best_iou}, CKPT)
print(f"training complete — best val IoU {best_iou:.3f} (weights: adas/drivable_idd_full_best.pth)")

In [ ]:
# 6) THE TEST — the same six dashcam frames (15000 is the one that matters)
import matplotlib.pyplot as plt

VIDEO_PATH = os.path.join(ADAS, "dashcam.mp4")
TEST_FRAMES = [2000, 5000, 9000, 12000, 15000, 17000]

model.load_state_dict(torch.load(BEST, map_location=DEVICE))
model.eval()

def infer_drivable(frame_bgr, in_w=768, in_h=432):
    h, w = frame_bgr.shape[:2]
    img = cv2.cvtColor(cv2.resize(frame_bgr, (in_w, in_h)), cv2.COLOR_BGR2RGB)
    x = (img.astype(np.float32) / 255.0 - MEAN) / STD
    x = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p = model(x)["out"].argmax(1).squeeze().cpu().numpy().astype(np.uint8)
    return cv2.resize(p, (w, h), interpolation=cv2.INTER_NEAREST)

cap = cv2.VideoCapture(VIDEO_PATH)
fig, axes = plt.subplots(len(TEST_FRAMES), 1, figsize=(14, 7 * len(TEST_FRAMES)))
for ax, fi in zip(axes, TEST_FRAMES):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ok, frame = cap.read()
    if not ok:
        continue
    da = infer_drivable(frame)
    color = np.zeros_like(frame); color[da == 1] = (0, 180, 0)
    vis = cv2.addWeighted(frame, 1.0, color, 0.45, 0)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"frame {fi} — full-IDD drivable area")
    ax.axis("off")
cap.release()
plt.tight_layout(); plt.show()

In [ ]:
# 7) Render the full hill segment + export ONNX for Jetson/OpenVINO
START, N_FRAMES = 14500, 900
OUT_PATH = os.path.join(ADAS, "phase6c_hills_segmented.mp4")

cap = cv2.VideoCapture(VIDEO_PATH)
fps_src = cap.get(cv2.CAP_PROP_FPS) or 30
W, H = int(cap.get(3)), int(cap.get(4))
cap.set(cv2.CAP_PROP_POS_FRAMES, START)
writer = cv2.VideoWriter(OUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps_src, (W, H))
for _ in tqdm(range(N_FRAMES)):
    ok, frame = cap.read()
    if not ok:
        break
    da = infer_drivable(frame)
    color = np.zeros_like(frame); color[da == 1] = (0, 180, 0)
    writer.write(cv2.addWeighted(frame, 1.0, color, 0.45, 0))
cap.release(); writer.release()
print("saved:", OUT_PATH)

dummy = torch.randn(1, 3, 432, 768).to(DEVICE)
torch.onnx.export(model, dummy, os.path.join(ADAS, "drivable_idd_full_768x432.onnx"),
                  input_names=["image"], output_names=["seg"], opset_version=12)
print("ONNX exported to Drive")

## If the session disconnects mid-training
Just reopen the notebook and **Runtime → Run all**. Cell 5 finds the checkpoint on Drive and resumes from the last finished epoch. The mask cache also lives on Drive, so only the 18 GB extraction (~10 min) repeats.

## Reading the results
- **Val drivable-IoU ≥ 0.90** is a strong model on Indian roads (full IDD usually gets there).
- **Frame 15000** — compare directly against the YOLOP result from Phase 6a.
- The ONNX file on Drive is the deployment artifact: TensorRT on a Jetson, or OpenVINO on the laptop's Intel iGPU.